In [ ]:
import pandas as pd 
import sys
from pathlib import Path
import re
import numpy as np
from nilearn import image, masking, plotting
from nilearn.maskers import NiftiMasker, NiftiLabelsMasker
from scipy.stats import (
    mode,
    pearsonr,
    spearmanr,
    false_discovery_control,
)

analysis_dir = Path("/projects/p32903/Alex2/ect-workflow/05_analysis")
if str(analysis_dir) not in sys.path: sys.path.insert(0, str(analysis_dir))
from simnibs_parcel_analysis.utils.loader import (
    load_query_results,
    load_course_data,
    load_stimulation_data,
    prepare_session_data,
)
from simnibs_parcel_analysis.utils.prepare_session_file_table import (
    MissingMappingError,
)



fn_query = '/projects/p32903/Alex2/ect-results-tools/query_result_2.csv'
fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest_20261029_1207.xlsx'

# atlases
fn_lut = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/7Networks_ColorLUT_freeview.txt'
fn_atl = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/Yeo2011_7Networks_N1000.split_components.FSL_MNI152_1mm.nii.gz'

## Prep

### ROI

In [ ]:
def load_scalar_img(niimg):
    """Load a 3D image, accepting a singleton fourth dimension."""
    img = image.load_img(niimg)

    if len(img.shape) == 4 and img.shape[-1] == 1:
        img = img.slicer[..., 0]

    if len(img.shape) != 3:
        raise ValueError(f"Expected 3D or (X,Y,Z,1); got {img.shape}")

    return img


def load_label_atlas(atlas_path, lut_path):
    """Return the atlas and a Series mapping label IDs to ROI names."""
    atlas = load_scalar_img(atlas_path)
    ids = np.unique(image.get_data(atlas))

    if not np.isfinite(ids).all() or not np.equal(ids, np.rint(ids)).all():
        raise ValueError("The atlas must contain finite integer labels.")

    ids = ids[ids != 0].astype(int)

    lut = pd.read_csv(
        lut_path,
        sep=r"\s+",
        comment="#",
        header=None,
        usecols=[0, 1],
        names=["id", "name"],
    ).set_index("id")["name"]

    if not lut.index.is_unique:
        raise ValueError("Duplicate IDs in the LUT.")

    names = lut.reindex(ids)

    if names.empty or names.isna().any() or not names.is_unique:
        raise ValueError("Every atlas ROI needs a unique name in the LUT.")

    return atlas, names


def extract_region_stats(df,
    atlas_path,
    lut_path,
    path_column="full_file_path",
    *,
    mode_decimals=None,):
    """Return {statistic: DataFrame}, preserving df's index and row order.

    Column names use one underscore between statistic and ROI:
    E_mean_7Networks_LH_Vis.

    Zeros are included. Non-finite E-fields raise an error.
    Missing regions receive count 0 and NaN for other statistics.
    mode_decimals affects only the mode calculation.
    """
    atlas, names = load_label_atlas(atlas_path, lut_path)

    statistics = (
        "mean", "mode", "median", "min", "max",
        "95th_percentile", "99th_percentile", "n_voxels",
    )

    shape = (len(df), len(names))
    arrays = {s: np.full(shape, np.nan) for s in statistics[:-1]}
    arrays["n_voxels"] = np.zeros(shape, dtype=np.int64)

    cached_grid, masker = None, None

    for row, path in enumerate(df[path_column]):
        img = load_scalar_img(path)

        if not np.isfinite(img.get_fdata()).all():
            raise ValueError(f"Non-finite E-field values: {path}")

        grid = (img.shape, tuple(img.affine.ravel()))

        if grid != cached_grid:
            # Align categorical labels to the E-field grid.
            labels = image.resample_to_img(
                atlas,
                img,
                interpolation="nearest",
                force_resample=True,
                copy_header=True,
            )

            roi_mask = image.math_img("labels != 0", labels=labels)
            cached_grid, masker = grid, None

            if image.get_data(roi_mask).any():
                masker = NiftiMasker(
                    mask_img=roi_mask,
                    dtype=np.float64,
                    standardize=None,
                    detrend=False,
                    smoothing_fwhm=None,
                    reports=False,
                ).fit()

                # Same voxel ordering as masker.transform().
                ids = masking.apply_mask(
                    labels, masker.mask_img_
                ).astype(int)

        if masker is None:
            continue

        values = pd.Series(np.asarray(masker.transform(img)).ravel())
        grouped = values.groupby(ids)

        summary = grouped.agg(
            mean="mean",
            median="median",
            min="min",
            max="max",
            n_voxels="size",
        )

        quantiles = grouped.quantile(
            [0.95, 0.99], interpolation="linear"
        ).unstack()

        summary["95th_percentile"] = quantiles[0.95]
        summary["99th_percentile"] = quantiles[0.99]

        mode_values = (
            values if mode_decimals is None
            else values.round(mode_decimals)
        )

        summary["mode"] = mode_values.groupby(ids).agg(
            lambda v: float(mode(v, axis=None, keepdims=False).mode)
        )

        summary = summary.reindex(names.index)
        summary["n_voxels"] = (
            summary["n_voxels"].fillna(0).astype(np.int64)
        )

        for stat in statistics:
            arrays[stat][row] = summary[stat].to_numpy()

    return {
        stat: pd.DataFrame(
            arrays[stat],
            index=df.index.copy(),
            columns=[f"E_{stat}_{name}" for name in names],
        )
        for stat in statistics
    }
    

def merge_series_data(df_cgi: pd.DataFrame, df_stim_collapsed: pd.DataFrame,) -> pd.DataFrame:
    keys = ["person_id", "series_num"]
    problems = []

    try:
        for name, table in [
            ("df_stim_collapsed", df_stim_collapsed),
            ("df_cgi", df_cgi),
        ]:
            missing_columns = set(keys) - set(table.columns)
            if missing_columns:
                raise KeyError(
                    f"{name} is missing columns: {sorted(missing_columns)}"
                )

            missing_keys = table.loc[table[keys].isna().any(axis=1)]
            if not missing_keys.empty:
                problems.append(f"{name} contains missing merge-key values")
                print(f"\n{name}: rows with missing merge keys:")
                print(missing_keys.to_string(index=True))

            duplicates = table.loc[table.duplicated(keys, keep=False)]
            if not duplicates.empty:
                problems.append(f"{name} contains duplicate series keys")
                print(f"\n{name}: rows with duplicate merge keys:")
                print(duplicates.to_string(index=True))

        stim_keys = pd.MultiIndex.from_frame(df_stim_collapsed[keys])
        cgi_keys = pd.MultiIndex.from_frame(df_cgi[keys])

        stim_only = df_stim_collapsed.loc[~stim_keys.isin(cgi_keys)]
        cgi_only = df_cgi.loc[~cgi_keys.isin(stim_keys)]

        if not stim_only.empty:
            problems.append(
                f"{len(stim_only)} stimulation rows have no matching CGI series"
            )
            print("\nStimulation rows without matching CGI series:")
            print(stim_only.to_string(index=True))

        if not cgi_only.empty:
            problems.append(
                f"{len(cgi_only)} CGI rows have no matching stimulation series"
            )
            print("\nCGI rows without matching stimulation series:")
            print(cgi_only.to_string(index=True))

        if problems:
            raise pd.errors.MergeError(
                "Perfect series alignment required: " + "; ".join(problems)
            )

        shared_metadata = [
            col for col in ["age", "sex", "dob"]
            if col in df_stim_collapsed.columns and col in df_cgi.columns
        ]

        return df_stim_collapsed.drop(columns=shared_metadata).merge(
            df_cgi,
            on=keys,
            how="left",
            sort=False,
            validate="one_to_one",
            suffixes=("_stim", "_cgi"),
        )

    except (KeyError, ValueError) as error:
        # pandas MergeError is also a ValueError.
        print(
            f"\nMerge failed: {error}\n"
            f"Stimulation rows: {len(df_stim_collapsed)}; "
            f"CGI rows: {len(df_cgi)}"
        )
        raise

def load_series_data(
    subjects_exclude: list,
    fn_query,
    fn_cgi,
    query_filters: dict,
    fn_field: str,
    *,
    subject_scan_map=None,
    session_num=2,) -> pd.DataFrame:
    # Load and filter the simulation inventory.
    df_query = load_query_results(fn_query)

    for column, value in query_filters.items():
        df_query = df_query.loc[df_query[column].eq(value)]

    df_query = df_query.loc[
        ~df_query["subject_dir"].isin(subjects_exclude)
    ].copy()

    if df_query.empty:
        raise ValueError(
            f"No query rows remain after filtering: {query_filters}"
        )

    # Load and filter the per-series outcomes.
    df_cgi = load_course_data(fn_cgi)
    df_cgi = df_cgi.loc[
        ~df_cgi["subjid"].isin(subjects_exclude)
    ].copy()

    # Select the representative session from each series.
    df_stim = load_stimulation_data(fn_cgi)
    df_stim = df_stim.loc[
        ~df_stim["subjid"].isin(subjects_exclude)
        & df_stim["person_id"].isin(df_cgi["person_id"])
        & df_stim["session_num"].eq(session_num)
    ].copy()

    if df_stim.empty:
        raise ValueError(
            f"No stimulation rows remain for session_num={session_num}."
        )

    # Temporary indices make fallback assignment independent of original indices.
    sessions = df_stim.reset_index(drop=True)

    mapping_options = dict(
        df_query=df_query,
        df_cgi=df_cgi,
        filename=fn_field,
        subject_scan_map=subject_scan_map,
    )

    # prepare_session_data also computes ef_sf.
    try:
        df_stim_mapped = prepare_session_data(
            df_stim=sessions,
            **mapping_options,
        )

    except MissingMappingError as error:
        placements = (
            error.unmatched["electrode_placement"]
            .str.strip()
            .str.upper()
        )

        if not placements.isin(["BT", "BL"]).all():
            raise

        fallback_sessions = sessions.copy()
        fallback_sessions.loc[
            error.unmatched.index, "electrode_placement"
        ] = placements.replace({"BT": "BL", "BL": "BT"})

        # Unresolved or ambiguous mappings still raise.
        df_stim_mapped = prepare_session_data(
            df_stim=fallback_sessions,
            **mapping_options,
        )

    # Restore the original clinical labels and indices.
    df_stim_mapped["electrode_placement"] = sessions.loc[
        df_stim_mapped.index, "electrode_placement"
    ]
    df_stim_mapped.index = df_stim.index.take(df_stim_mapped.index)

    # Uses the strict alignment checks and diagnostics defined earlier.
    return merge_series_data(
        df_cgi=df_cgi,
        df_stim_collapsed=df_stim_mapped,
    )



### Session Tables

In [ ]:


subjects_exclude = ["subj-cat-010-001"]
fn_field = "tdcs_uq_gpc_MNI_magnE_mean.nii.gz"

common_filters = {
    "threshold": 95,
    "step_result": "mni_volumes",
    "asf": "asf_5",
}

series_tables = {}

for model_type in ("skin_single", "skin_double"):
    for step in ("step_0", "step_9"):
        query_filters = {
            **common_filters,
            "model_type_dir": model_type,
            "step": step,
        }

        series_tables[(model_type, step)] = load_series_data(
            subjects_exclude=subjects_exclude,
            fn_query=fn_query,
            fn_cgi=fn_cgi,
            query_filters=query_filters,
            fn_field=fn_field,
            subject_scan_map=analysis_dir / "subject_scan_map.json",
            session_num=2,
        )

# df_skin_single_step0 = series_tables[("skin_single", "step_0")]
# df_skin_single_step9 = series_tables[("skin_single", "step_9")]
# df_skin_double_step0 = series_tables[("skin_double", "step_0")]
# df_skin_double_step9 = series_tables[("skin_double", "step_9")]

# Save after all four tables have been built successfully.
output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)


# todo: this step could be removed, I don't see the point in having this. 
for (model_type, step), df_merged in series_tables.items():
    output_name = (
        "series_magnE_mean_MNI152_session2_"
        f"{step.replace('_', '')}_{model_type}.csv"
    )
    df_merged.to_csv(output_dir / output_name, index=False)

### ROI + sessions

In [ ]:
from pathlib import Path
import pandas as pd

data_dir = Path("data")
series_tables_with_roi = {}
fn_lut = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/7Networks_ColorLUT_freeview.txt'
fn_atl = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/Yeo2011_7Networks_N1000.split_components.FSL_MNI152_1mm.nii.gz'

for model_type in ("skin_single", "skin_double"):
    for step in ("step_0", "step_9"):
        stem = (
            "series_magnE_mean_MNI152_session2_"
            f"{step.replace('_', '')}_{model_type}"
        )

        df_merged = pd.read_csv(data_dir / f"{stem}.csv")

        # Different treatment series can reference the same simulation file.
        unique_fields = (
            df_merged[["full_file_path"]]
            .drop_duplicates()
            .reset_index(drop=True)
        )

        roi_stats = extract_region_stats(
            unique_fields,
            atlas_path=fn_atl,
            lut_path=fn_lut,
            path_column="full_file_path",
            mode_decimals=None,
        )

        field_stats = pd.concat(
            [unique_fields, *roi_stats.values()],
            axis=1,
        )

        # Attach the statistics to every series using that file.
        df_results = df_merged.merge(
            field_stats,
            on="full_file_path",
            how="left",
            sort=False,
            validate="many_to_one",
        )

        series_tables_with_roi[(model_type, step)] = df_results

        output_path = data_dir / f"{stem}_yeo2011_7network_ROI.csv"
        df_results.to_csv(output_path, index=False)

        print(
            f"{model_type}, {step}: "
            f"{len(df_results)} series, "
            f"{len(unique_fields)} unique files → {output_path}"
        )

# Access each resulting dataframe.
# df_skin_single_step0_roi = series_tables_with_roi[("skin_single", "step_0")]
# df_skin_single_step9_roi = series_tables_with_roi[("skin_single", "step_9")]
# df_skin_double_step0_roi = series_tables_with_roi[("skin_double", "step_0")]
# df_skin_double_step9_roi = series_tables_with_roi[("skin_double", "step_9")]

tbl_shape   = series_tables_with_roi[("skin_single", "step_0")].shape
n_rois      = series_tables_with_roi[("skin_single", "step_0")].columns.str.contains('E_mean').sum()
print(f"Series tables w/ atlas ROIS (n_rois={n_rois}). Total shape: {tbl_shape}")


def save_series_roi_master( tables: dict, filename: str | Path) -> pd.DataFrame:
    """Combine all series/ROI tables, add configuration labels, and save CSV."""
    if not tables:
        raise ValueError("No series/ROI tables supplied.")

    master = pd.concat(
        [
            df.assign(model_type=model_type, step=step)
            for (model_type, step), df in tables.items()
        ],
        ignore_index=True,
    )

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    master.to_csv(output_path, index=False)

    print(
        f"Saved {len(master)} rows and "
        f"{len(master.columns)} columns to {output_path}"
    )

    return master


# save into a large file with all clinical and cgi information
df_series_roi_master = save_series_roi_master(
    series_tables_with_roi,
    "data/yeo2011_series_with_ROI_all_configurations.csv",
)

## Run:

### Correlations cgi_change ~ magnE_mean(ROI)

In [ ]:
from pathlib import Path
import pandas as pd
import pandas as pd
from scipy.stats import spearmanr

def keep_cgi_and_field(df: pd.DataFrame) -> pd.DataFrame:
    mean_columns = [
        col for col in df.columns
        if col.startswith("E_mean_7Networks_")
    ]

    if not mean_columns:
        raise ValueError("No E_mean_ columns found.")

    return df.loc[:, ["cgi_change", *mean_columns]].copy()


def add_network_means_both_hemis(df: pd.DataFrame) -> pd.DataFrame:
    networks = [
        "Vis", "SomMot", "DorsAttn", "SalVentAttn",
        "Limbic", "Cont", "Default",
    ]

    result = df.copy()

    for network in networks:
        columns = []

        for hemisphere in ("LH", "RH"):
            prefix = f"E_mean_7Networks_{hemisphere}_{network}"

            region_columns = [
                col for col in df.columns
                if col == prefix or col.startswith(prefix + "_")
            ]

            if not region_columns:
                raise ValueError(
                    f"No regional mean columns for {hemisphere} {network}."
                )

            columns.extend(region_columns)

        result[f"E_mean_{network}_both_hemis"] = df[columns].mean(
            axis=1,
            skipna=False,
        )

    return result


def save_field_roi_tables(tables: dict, filename) -> pd.DataFrame:
    if not tables:
        raise ValueError("No feature tables supplied.")

    combined = pd.concat(
        [
            df.assign(model_type=model_type, step=step)
            for (model_type, step), df in tables.items()
        ],
        ignore_index=True,
    )

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(output_path, index=False)

    print(
        f"Saved {len(combined)} rows and "
        f"{len(combined.columns)} columns to {output_path}"
    )

    return combined


def run_spearman(df: pd.DataFrame, y: str, x: list[str] | str | None = None,) -> pd.DataFrame:
    if df.columns.has_duplicates:
        raise ValueError("Dataframe contains duplicate column names.")

    if y not in df.columns:
        raise KeyError(f"Outcome column {y!r} is missing.")

    if x is None:
        features = [col for col in df.columns if col != y]
    elif isinstance(x, str):
        features = [x]
    else:
        features = list(x)

    if not features:
        raise ValueError("No feature columns selected.")

    if y in features or len(features) != len(set(features)):
        raise ValueError("Features must be unique and must not include y.")

    data = df[[y, *features]].apply(pd.to_numeric, errors="raise")

    if len(data) < 3:
        raise ValueError("At least three observations are required.")

    invalid = data.columns[
        ~np.isfinite(data.to_numpy(dtype=float)).all(axis=0)
    ].tolist()
    if invalid:
        raise ValueError(f"Missing or non-finite values in: {invalid}")

    constant = data.columns[data.nunique().lt(2)].tolist()
    if constant:
        raise ValueError(f"Constant columns cannot be correlated: {constant}")

    results = []
    for feature in features:
        rho, p_value = spearmanr(
            data[feature],
            data[y],
            nan_policy="raise",
        )
        results.append({
            "x": feature,
            "y": y,
            "n": len(data),
            "rho": float(rho),
            "p_value": float(p_value),
        })

    return pd.DataFrame(results)


def save_spearman_table(df: pd.DataFrame, filename) -> pd.DataFrame:
    columns = ["x", "y", "n", "rho", "p_value", "model_type", "step"]

    missing = set(columns) - set(df.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    result = df[columns].copy()

    networks = "Vis|SomMot|DorsAttn|SalVentAttn|Limbic|Cont|Default"

    # Single underscore after E_mean.
    regional_pattern = (
        rf"E_mean_7Networks_(LH|RH)_({networks})(?:_(.+))?"
    )
    bilateral_pattern = rf"E_mean_({networks})_both_hemis"

    metadata = []

    for feature in result["x"]:
        if not isinstance(feature, str):
            raise ValueError(f"Invalid feature name: {feature!r}")

        regional = re.fullmatch(regional_pattern, feature)
        bilateral = re.fullmatch(bilateral_pattern, feature)

        if regional:
            hemi, network, roi = regional.groups()
            roi = roi or network
        elif bilateral:
            network = bilateral.group(1)
            hemi = "both"
            roi = network
        else:
            raise ValueError(f"Unrecognized feature name: {feature!r}")

        metadata.append((roi, hemi, network))

    result["roi"] = [row[0] for row in metadata]
    result["hemi"] = [row[1] for row in metadata]
    result["network"] = [row[2] for row in metadata]

    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    result.to_csv(output_path, index=False)

    return result


# collapse to N_networks X hemisphere
cgi_and_feature_tables = {
    configuration: keep_cgi_and_field(df)
    for configuration, df in series_tables_with_roi.items()
}

tbl_shape   = cgi_and_feature_tables[("skin_single", "step_0")].shape
n_rois      = cgi_and_feature_tables[("skin_single", "step_0")].columns.str.contains('E_mean').sum()
print(f"Series tables w/ atlas ROIS (n_rois={n_rois}) and `cgi_change` column. Total shape: {tbl_shape}")
# df_skin_single_step0_corr = cgi_and_feature_tables[("skin_single", "step_0")]
# df_skin_single_step9_corr = cgi_and_feature_tables[("skin_single", "step_9")]
# df_skin_double_step0_corr = cgi_and_feature_tables[("skin_double", "step_0")]
# df_skin_double_step9_corr = cgi_and_feature_tables[("skin_double", "step_9")]

# Append network columns to the existing correlation tables.
correlation_tables_both_hemis = {
    configuration: add_network_means_both_hemis(df)
    for configuration, df in cgi_and_feature_tables.items()
}

# tbl_shape   = correlation_tables_both_hemis[("skin_single", "step_0")].shape
# n_rois      = correlation_tables_both_hemis[("skin_single", "step_0")].columns.str.contains('E_mean').sum()
# print("Adding feature columns: Per-network mean across both hemispheres.")
# print(f"Series tables w/ atlas ROIS (n_rois={n_rois}) and `cgi_change` column. Total shape: {tbl_shape}")

# df_field_rois_saved = save_field_roi_tables(
#     correlation_tables_both_hemis,
#     "data/yeo2011_field_ROIs_for_spearman_all_configurations.csv",
# )

# # df_skin_single_step0_corr = correlation_tables_both_hemis[("skin_single", "step_0")]
# # df_skin_single_step9_corr = correlation_tables_both_hemis[("skin_single", "step_9")]
# # df_skin_double_step0_corr = correlation_tables_both_hemis[("skin_double", "step_0")]
# # df_skin_double_step9_corr = correlation_tables_both_hemis[("skin_double", "step_9")]

# # # All features for all four configurations.
# # spearman_results = {
# #     configuration: run_spearman(df, y="cgi_change")
# #     for configuration, df in correlation_tables_both_hemis.items()
# # }

# # df_spearman_all = pd.concat(
# #     [
# #         results.assign(model_type=model_type, step=step)
# #         for (model_type, step), results in spearman_results.items()
# #     ],
# #     ignore_index=True,
# # )

# # df_spearman_annotated = save_spearman_table(df_spearman_all, "data/spearman_yeo2011_ROI_and_network_annotated.csv")

In [ ]:
correlation_tables_both_hemis[("skin_single", "step_0")]

### Plot Results

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from nilearn import image, plotting


def network_correlations_to_image(
    results,
    atlas_path,
    lut_path,
    *,
    level="network",
):
    """Map either 'roi' or 'network' correlations onto the atlas."""
    if level not in ("roi", "network"):
        raise ValueError("level must be 'roi' or 'network'.")

    results = results.copy()

    # Normalize names from older saved correlation tables.
    results["x"] = results["x"].str.replace(
        r"^E_mean__7Networks_",
        "E_mean_7Networks_",
        regex=True,
    )

    if results["x"].duplicated().any():
        raise ValueError("Duplicate features: select one configuration only.")

    if results["y"].nunique(dropna=False) != 1:
        raise ValueError("Results must contain exactly one outcome.")

    atlas, names = load_label_atlas(atlas_path, lut_path)
    labels = image.get_data(atlas)

    networks = names.str.extract(
        r"^7Networks_(?:LH|RH)_([^_]+)(?:_|$)",
        expand=False,
    )

    if networks.isna().any():
        raise ValueError(
            f"Unrecognized atlas labels: {names[networks.isna()].tolist()}"
        )

    # Each Series maps atlas label IDs to correlation feature names.
    roi_features = "E_mean_" + names
    network_features = "E_mean_" + networks + "_both_hemis"

    known_features = set(roi_features) | set(network_features)
    unexpected = set(results["x"]) - known_features

    if unexpected:
        raise ValueError(f"Unrecognized features: {sorted(unexpected)}")

    features = roi_features if level == "roi" else network_features
    required = set(features)

    selected = results.loc[results["x"].isin(required)]
    missing = required - set(selected["x"])

    if missing:
        raise ValueError(f"Missing {level} features: {sorted(missing)}")

    rho = pd.to_numeric(
        selected.set_index("x")["rho"],
        errors="raise",
    )

    if not np.isfinite(rho.to_numpy(dtype=float)).all():
        raise ValueError("Correlation coefficients contain non-finite values.")

    if rho.abs().gt(1).any():
        raise ValueError("Correlation coefficients must be within [-1, 1].")

    rho_volume = np.zeros(labels.shape, dtype=np.float32)

    for label_id, feature in features.items():
        rho_volume[labels == label_id] = rho.loc[feature]

    rho_img = image.new_img_like(atlas, rho_volume, copy_header=True)
    rho_img.set_data_dtype(np.float32)

    return rho_img

all_results = pd.read_csv(
    "data/spearman_yeo2011_ROI_and_network_annotated.csv"
)

mni_background = image.load_img(
    "/projects/p32903/Alex2/simnibs_dev/simnibs/simnibs/"
    "resources/templates/MNI152_T1_1mm.nii.gz"
)

output_dir = Path("data/spearman_maps")
output_dir.mkdir(parents=True, exist_ok=True)

rho_maps = {}

for (model_type, step), results in all_results.groupby(
    ["model_type", "step"],
    sort=False,
):
    for level in ("roi", "network"):
        rho_img = network_correlations_to_image(
            results,
            atlas_path=fn_atl,
            lut_path=fn_lut,
            level=level,
        )

        rho_maps[(model_type, step, level)] = rho_img
        stem = f"spearman_yeo2011_{model_type}_{step}_{level}"

        rho_img.to_filename(output_dir / f"{stem}_rho.nii.gz")

        display = plotting.plot_stat_map(
            rho_img,
            bg_img=mni_background,
            display_mode="z",
            cut_coords=[-20, 0, 20, 40, 60],
            threshold=0,
            cmap="RdBu_r",
            vmin=-1,
            vmax=1,
            symmetric_cbar=True,
            resampling_interpolation="nearest",
            colorbar=True,
            cbar_tick_format="%.2f",
            title=f"{model_type} | {step} | {level} | Spearman rho",
        )

        display.savefig(output_dir / f"{stem}_rho.png", dpi=300)

# plotting.show()

## test

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from nilearn import image, plotting


def plot_save_parent_network(
    results,
    *,
    network,
    model_type,
    step,
    atlas_path,
    lut_path,
    template_path,
    output_dir,
    y="cgi_change",
):
    """Save regional and bilateral-network maps for one parent/configuration."""
    atlas, names = load_label_atlas(atlas_path, lut_path)
    labels = image.get_data(atlas)

    # Select this parent's atlas regions, keeping both hemispheres.
    roi_names = names.loc[
        names.str.split("_").str[2].eq(network)
    ]

    hemispheres = set(roi_names.str.split("_").str[1])
    if roi_names.empty or hemispheres != {"LH", "RH"}:
        raise ValueError(
            f"Atlas must contain LH and RH regions for {network!r}."
        )

    roi_features = ("E_mean_" + roi_names).tolist()
    bilateral_feature = f"E_mean_{network}_both_hemis"
    required = [*roi_features, bilateral_feature]

    # Select one configuration and outcome.
    selected = results.loc[
        results["model_type"].eq(model_type)
        & results["step"].eq(step)
        & results["y"].eq(y)
    ].copy()

    # Support older CSVs; normalized duplicate names still raise below.
    selected["x"] = selected["x"].str.replace(
        r"^E_mean__7Networks_",
        "E_mean_7Networks_",
        regex=True,
    )

    selected = selected.loc[selected["x"].isin(required)]

    if selected["x"].duplicated().any():
        duplicates = selected.loc[
            selected["x"].duplicated(keep=False), "x"
        ].tolist()
        raise ValueError(f"Duplicate correlation features: {duplicates}")

    missing = set(required) - set(selected["x"])
    if missing:
        raise ValueError(
            f"{model_type}/{step}/{network}: "
            f"missing correlation features: {sorted(missing)}"
        )

    selected = selected.set_index("x").loc[required].copy()
    rho = pd.to_numeric(selected["rho"], errors="raise")

    if not np.isfinite(rho.to_numpy(dtype=float)).all():
        raise ValueError("Non-finite correlation coefficients.")

    if rho.abs().gt(1).any():
        raise ValueError("Correlation coefficients must be within [-1, 1].")

    selected["rho"] = rho

    parent_mask = np.isin(labels, roi_names.index)
    regional_volume = np.zeros(labels.shape, dtype=np.float32)

    # Choose a representative axial slice for every component ROI.
    # Both map types use these same deterministic slice coordinates.
    cuts = []

    for label_id, roi_name in roi_names.items():
        roi_mask = labels == label_id
        regional_volume[roi_mask] = rho.loc[f"E_mean_{roi_name}"]

        k = int(roi_mask.sum(axis=(0, 1)).argmax())
        i, j = np.argwhere(roi_mask[:, :, k])[0]
        world_z = (atlas.affine @ [i, j, k, 1])[2]
        cuts.append(float(world_z))

    cuts = sorted(set(cuts))

    bilateral_volume = np.zeros(labels.shape, dtype=np.float32)
    bilateral_volume[parent_mask] = rho.loc[bilateral_feature]

    # Transparency hides everything outside this parent network,
    # while allowing a genuine rho=0 region to remain visible.
    mask_img = image.new_img_like(
        atlas,
        parent_mask.astype(np.uint8),
        copy_header=False,
    )
    background = image.load_img(template_path)

    jobs = [
        (
            "regional_ROIs_LH_RH_separate",
            regional_volume,
            roi_features,
            "Each LH/RH region has its own correlation coefficient",
        ),
        (
            "network_mean_both_hemis",
            bilateral_volume,
            [bilateral_feature],
            "One network coefficient: regional E means combined across LH + RH",
        ),
    ]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}

    for tag, volume, features, description in jobs:
        prefix = (
            f"spearman_rho_{model_type}_{step}_{y}_"
            f"parent_{network}_{tag}"
        )

        paths = {
            "nifti": output_dir / f"{prefix}.nii.gz",
            "png": output_dir / f"{prefix}.png",
            "csv": output_dir / f"{prefix}.csv",
        }

        rho_img = image.new_img_like(atlas, volume, copy_header=True)
        rho_img.set_data_dtype(np.float32)
        rho_img.header["cal_min"] = -1
        rho_img.header["cal_max"] = 1
        rho_img.to_filename(paths["nifti"])

        # Regional CSV: one row per component ROI.
        # Bilateral CSV: exactly one network-level correlation row.
        selected.loc[features].reset_index().to_csv(
            paths["csv"],
            index=False,
        )

        figure = plt.figure(
            figsize=(max(10, 2 * len(cuts) + 1), 4)
        )

        display = plotting.plot_stat_map(
            rho_img,
            bg_img=background,
            figure=figure,
            display_mode="z",
            cut_coords=cuts,
            threshold=None,
            transparency=mask_img,
            transparency_range=[0, 1],
            resampling_interpolation="nearest",
            cmap="RdBu_r",
            vmin=-1,
            vmax=1,
            symmetric_cbar=True,
            colorbar=True,
            cbar_tick_format="%.2f",
            title=(
                f"Parent network: {network} | {model_type} | {step}\n"
                f"{description}\nSpearman rho vs {y}"
            ),
        )

        display.savefig(paths["png"], dpi=300)
        display.close()

        saved[tag] = paths

    return saved

all_results = pd.read_csv(
    "data/spearman_yeo2011_ROI_and_network_annotated.csv"
)

fn_mni = (
    "/projects/p32903/Alex2/simnibs_dev/simnibs/simnibs/"
    "resources/templates/MNI152_T1_1mm.nii.gz"
)

networks = [
    "Vis", "SomMot", "DorsAttn", "SalVentAttn",
    "Limbic", "Cont", "Default",
]

saved_plots = {}

for model_type in ("skin_single", "skin_double"):
    for step in ("step_0", "step_9"):
        for network in networks:
            saved_plots[(model_type, step, network)] = (
                plot_save_parent_network(
                    all_results,
                    network=network,
                    model_type=model_type,
                    step=step,
                    atlas_path=fn_atl,
                    lut_path=fn_lut,
                    template_path=fn_mni,
                    output_dir="data/spearman_parent_network_maps",
                    y="cgi_change",
                )
            )

print(f"Saved {2 * len(saved_plots)} NIfTI/PNG/CSV sets.")


# notes

In [ ]:
from pathlib import Path
import pandas as pd
import re


def get_available_steps(df_query: pd.DataFrame, query_filters: dict, fn_field: str, subj_exclude: list | None = None) -> list:
    """Find available steps after field, configuration, and subject filtering."""
    excluded = [] if subj_exclude is None else subj_exclude
    selected = df_query

    for column, value in query_filters.items():
        selected = selected.loc[selected[column].eq(value)]

    selected = selected.loc[
        ~selected["subject_dir"].isin(excluded)
    ]

    selected = selected.loc[
        selected["full_file_path"].map(
            lambda path: Path(path).name == fn_field
        )
    ]

    available = selected["step"].unique().tolist()

    if any(not isinstance(step, str) or not re.fullmatch(r"step_\d+", step) for step in available):
        raise ValueError(
            f"Invalid step labels in matching query rows: {available}"
        )

    # Numeric ordering: step_0, step_2, step_9, step_10.
    return sorted(
        available,
        key=lambda step: (int(step.split("_", 1)[1]), step),
    )


def load_series_tables(fn_query, fn_cgi, subject_scan_map, field_name: str = "magnE_mean", steps: list | None = None, subj_exclude: list | None = None) -> dict:
    """Load requested steps, or discover available steps separately per model."""
    excluded = [] if subj_exclude is None else subj_exclude

    if steps is not None:
        if not isinstance(steps, list) or not steps:
            raise ValueError(
                "steps must be None or a nonempty list of step labels."
            )

        if any(not isinstance(step, str) or not re.fullmatch(r"step_\d+", step) for step in steps):
            raise ValueError(
                "Use step labels such as 'step_0' and 'step_9'."
            )

        if len(steps) != len(set(steps)):
            raise ValueError("steps contains duplicate labels.")

    fn_field = f"tdcs_uq_gpc_MNI_{field_name}.nii.gz"
    df_query = load_query_results(fn_query)

    common_filters = {
        "threshold": 95,
        "step_result": "mni_volumes",
        "asf": "asf_5",
    }

    # Validate the requested configurations before loading clinical data.
    configurations = []

    for model_type in ("skin_single", "skin_double"):
        filters = {
            **common_filters,
            "model_type_dir": model_type,
        }

        available = get_available_steps(
            df_query,
            filters,
            fn_field,
            subj_exclude=excluded,
        )

        if steps is None:
            selected_steps = available
        else:
            missing = set(steps) - set(available)

            if missing:
                raise ValueError(
                    f"{model_type}: requested steps have no matching field files: "
                    f"{sorted(missing)}. Available steps: {available}"
                )

            selected_steps = steps

        print(f"{model_type}: selected steps = {selected_steps}")

        configurations.extend(
            (model_type, step, filters)
            for step in selected_steps
        )

    if not configurations:
        raise ValueError(
            "No matching model/step configurations remain after filtering."
        )

    tables = {}

    for model_type, step, filters in configurations:
        table = load_series_data(
            subjects_exclude=excluded,
            fn_query=fn_query,
            fn_cgi=fn_cgi,
            query_filters={**filters, "step": step},
            fn_field=fn_field,
            subject_scan_map=subject_scan_map,
            session_num=2,
        )

        tables[(model_type, step)] = table.assign(
            model_type=model_type,
            step=step,
            field_name=field_name,
        )

    return tables


def build_roi_metadata(atlas_path, lut_path) -> pd.DataFrame:
    """Get ROI names and their hemisphere/network labels."""
    _, names = load_label_atlas(atlas_path, lut_path)

    metadata = pd.DataFrame({"roi_name": names.to_numpy()})
    metadata[["hemi", "network"]] = metadata["roi_name"].str.extract(
        r"^7Networks_(LH|RH)_([^_]+)(?:_.+)?$"
    )

    if metadata[["hemi", "network"]].isna().any().any():
        raise ValueError("Unrecognized ROI names in the atlas LUT.")

    return metadata


def extract_unique_field_stats(series_tables: dict, atlas_path, lut_path) -> tuple:
    """Extract statistics once per unique file across all configurations."""
    if not series_tables:
        raise ValueError("No series tables supplied.")

    unique_fields = (
        pd.concat(
            [table[["full_file_path"]] for table in series_tables.values()],
            ignore_index=True,
        )
        .drop_duplicates()
        .reset_index(drop=True)
    )

    if unique_fields.empty:
        raise ValueError("No simulation files to extract.")

    roi_stats = extract_region_stats(
        unique_fields,
        atlas_path=atlas_path,
        lut_path=lut_path,
        path_column="full_file_path",
        mode_decimals=None,
    )

    return unique_fields, roi_stats


def roi_statistics_to_rows(unique_fields: pd.DataFrame, roi_stats: dict, roi_metadata: pd.DataFrame) -> pd.DataFrame:
    """Convert metric-specific ROI columns to file/ROI rows."""
    metrics = {
        "mean": "mean",
        "mode": "mode",
        "median": "median",
        "min": "min",
        "max": "max",
        "95th_percentile": "p95",
        "99th_percentile": "p99",
        "n_voxels": "n_voxels",
    }

    rows = unique_fields.merge(
        roi_metadata[["roi_name"]],
        how="cross",
    )

    for source_metric, output_metric in metrics.items():
        table = roi_stats[source_metric]
        expected = [
            f"E_{source_metric}_{roi}"
            for roi in roi_metadata["roi_name"]
        ]

        if not table.index.equals(unique_fields.index):
            raise ValueError(
                f"Unexpected row alignment for {source_metric}."
            )

        if table.columns.has_duplicates or set(table.columns) != set(expected):
            raise ValueError(
                f"Unexpected ROI columns for {source_metric}."
            )

        rows[output_metric] = table[expected].to_numpy().ravel()

    return rows


def merge_series_with_rois(series_tables: dict, field_rows: pd.DataFrame, roi_metadata: pd.DataFrame) -> pd.DataFrame:
    """Attach ROI statistics to every series, preserving clinical metadata."""
    if not series_tables:
        raise ValueError("No series tables supplied.")

    metric_columns = set(field_rows.columns) - {"full_file_path", "roi_name"}
    reserved = set(roi_metadata.columns) | metric_columns | {"_roi_join"}
    tables = []

    for configuration, series in series_tables.items():
        overlap = reserved.intersection(series.columns)

        if overlap:
            raise ValueError(
                f"{configuration}: clinical columns conflict with ROI output: "
                f"{sorted(overlap)}"
            )

        expanded = series.merge(roi_metadata, how="cross")

        table = expanded.merge(
            field_rows,
            on=["full_file_path", "roi_name"],
            how="left",
            sort=False,
            validate="many_to_one",
            indicator="_roi_join",
        )

        if table["_roi_join"].ne("both").any():
            raise ValueError(
                f"Missing ROI-statistic matches for {configuration}."
            )

        tables.append(table.drop(columns="_roi_join"))

    master = pd.concat(tables, ignore_index=True)

    keys = [
        "person_id", "series_num", "model_type",
        "step", "field_name", "roi_name",
    ]

    if master.duplicated(keys).any():
        raise ValueError(
            "Duplicate series/configuration/field/ROI rows."
        )

    first = [
        "person_id", "series_num", "model_type", "step", "field_name",
        "roi_name", "hemi", "network",
        "n_voxels", "mean", "mode", "median", "min", "max", "p95", "p99",
        "cgi_start", "cgi_end", "cgi_change",
    ]

    return master[
        first + [col for col in master.columns if col not in first]
    ]


def build_series_roi_master(fn_query, fn_cgi, atlas_path, lut_path, subject_scan_map, field_name: str = "magnE_mean", steps: list | None = None, subj_exclude: list | None = None) -> pd.DataFrame:
    """Coordinate loading, extraction, reshaping, and joining."""
    series_tables = load_series_tables(
        fn_query=fn_query,
        fn_cgi=fn_cgi,
        subject_scan_map=subject_scan_map,
        field_name=field_name,
        steps=steps,
        subj_exclude=subj_exclude,
    )

    roi_metadata = build_roi_metadata(atlas_path, lut_path)

    unique_fields, roi_stats = extract_unique_field_stats(
        series_tables, atlas_path, lut_path
    )

    field_rows = roi_statistics_to_rows(
        unique_fields, roi_stats, roi_metadata
    )

    return merge_series_with_rois(
        series_tables, field_rows, roi_metadata
    )


def parse_roi_metadata(df: pd.DataFrame) -> pd.DataFrame:
    """Preserve the original atlas name and parse its component labels."""
    if df.columns.has_duplicates:
        raise ValueError("Dataframe contains duplicate column names.")

    if "roi_name" in df.columns and "roi_org" in df.columns:
        raise ValueError("Both roi_name and roi_org exist; resolve before parsing.")

    result = df.rename(columns={"roi_name": "roi_org"}).copy()

    if "roi_org" not in result.columns:
        raise KeyError("Expected a roi_name or roi_org column.")

    parsed = result["roi_org"].str.extract(
        r"^7Networks_(?P<hemi>LH|RH)_(?P<network>[^_]+)(?:_(?P<roi>.+))?$"
    )

    invalid = parsed[["hemi", "network"]].isna().any(axis=1)
    if invalid.any():
        names = result.loc[invalid, "roi_org"].unique().tolist()
        raise ValueError(f"Unrecognized ROI names: {names}")

    # Verify existing labels rather than silently replacing conflicting values.
    for column in ("hemi", "network"):
        if column in result.columns:
            conflict = (
                result[column].notna()
                & result[column].ne(parsed[column])
            )
            if conflict.any():
                raise ValueError(
                    f"Existing {column} values conflict with roi_org."
                )

        result[column] = parsed[column]

    result["roi"] = parsed["roi"].fillna(parsed["network"])

    # Place the short label immediately after the original name.
    columns = [column for column in result.columns if column != "roi"]
    columns.insert(columns.index("roi_org") + 1, "roi")

    return result[columns]


def save_roi_master(master: pd.DataFrame, filename: str | Path) -> Path:
    """Save an already-built master dataframe."""
    output_path = Path(filename)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    master.to_csv(output_path, index=False)

    print(
        f"Saved {len(master)} rows and "
        f"{len(master.columns)} columns to {output_path}"
    )

    return output_path



In [ ]:
field_name = "magnE_mean"

fn_query = '/projects/p32903/Alex2/ect-results-tools/query_result_2.csv'
fn_cgi   = '/projects/p32903/Alex2/datasets/STU00225089/.clincal_notes/cgi_batch_latest_20261029_1207.xlsx'

# atlases
fn_lut = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/7Networks_ColorLUT_freeview.txt'
fn_atl = '/projects/p32903/Alex2/ect-workflow/atlases/Yeo2011/Yeo2011_7Networks_N1000.split_components.FSL_MNI152_1mm.nii.gz'

steps = ["step_0", "step_9"]    # Or ["step_0", "step_9"].
subj_exclude = ['subj-cat-010-001']  # Add subject IDs to exclude here.

df_series_roi_master = build_series_roi_master(
    fn_query=fn_query,
    fn_cgi=fn_cgi,
    atlas_path=fn_atl,
    lut_path=fn_lut,
    subject_scan_map=analysis_dir / "subject_scan_map.json",
    field_name=field_name,
    steps=steps,
    subj_exclude=subj_exclude,
)

df_series_roi_master = parse_roi_metadata(df_series_roi_master)

saved_path = save_roi_master(
    df_series_roi_master,
    f"data/yeo2011_{field_name}_series_ROI_master.csv",
)

skin_single: selected steps = ['step_0', 'step_9']
skin_double: selected steps = ['step_0', 'step_9']
Saved 7140 rows and 56 columns to data/yeo2011_magnE_mean_series_ROI_master.csv
Saved 7140 rows and 57 columns to data/yeo2011_magnE_mean_series_ROI_master.csv
